# Notebook 2: Shape Tracking, Parameter Counting and Receptive Fields

**Goal.** Never again see `RuntimeError: mat1 and mat2 shapes cannot be multiplied`. By the end you will be able to predict, by hand, the tensor shape after every layer of a CNN, its parameter count, its compute cost, and how much of the input image each neuron can "see".

**What you will learn**

1. The output-size formula for convolution and pooling, including dilation, and where it comes from.
2. Three practical ways to inspect shapes: printing in `forward`, a manual layer walker, and forward hooks.
3. Counting parameters and multiply-accumulate operations (MACs).
4. The **receptive field**: definition, recursive formula, and empirical measurement via gradients.
5. Tricks that remove hard-coded sizes: `nn.LazyLinear`, `nn.AdaptiveAvgPool2d`, global average pooling.
6. The most common shape bugs and how to read their error messages.

This notebook needs no dataset and runs in seconds on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
torch.manual_seed(0)

## 1. Deriving the output-size formula

Consider one spatial dimension of length $H$. Pad it with $p$ zeros on each side, giving length $H + 2p$. A kernel of size $k$ fits in positions $0, 1, \dots, H + 2p - k$ (its left edge). With stride $s$ we only use every $s$-th of these positions, so the number of valid placements is

$$
H_\text{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1.
$$

**Dilation** $d$ spreads the kernel taps apart by $d$ pixels (inserting $d-1$ holes between them). A dilated kernel covers an *effective* size $k_\text{eff} = d(k-1) + 1$, so in general

$$
H_\text{out} = \left\lfloor \frac{H + 2p - d(k-1) - 1}{s} \right\rfloor + 1.
$$

This is exactly the formula in the PyTorch documentation for `nn.Conv2d`, and `nn.MaxPool2d` / `nn.AvgPool2d` use the same one (their default stride equals `kernel_size`).

### Useful special cases to memorise

| Setting | Effect |
|---|---|
| $k=3, p=1, s=1$ | size preserved ("same") |
| $k=5, p=2, s=1$ | size preserved |
| general odd $k$, $p=(k-1)/2$, $s=1$ | size preserved |
| $k=3, p=1, s=2$ | size halved (rounding up: $\lceil H/2 \rceil$) |
| MaxPool $k=2, s=2$ | size halved (rounding down) |
| $k=1$ | size preserved, mixes channels only ("pointwise" conv) |

The floor is important: a $7\times7$ map through `MaxPool2d(2)` becomes $3\times3$, silently discarding the last row and column.

In [ ]:
def conv_out(h, k, s=1, p=0, d=1):
    return (h + 2 * p - d * (k - 1) - 1) // s + 1

# Brute-force check of the formula against PyTorch for many configurations
mismatches = 0
for h in [7, 8, 28, 32, 33]:
    for k in [1, 2, 3, 5]:
        for s in [1, 2, 3]:
            for p in [0, 1, 2]:
                for d in [1, 2]:
                    if p > (d * (k - 1) + 1) // 2 or conv_out(h, k, s, p, d) < 1:
                        continue
                    real = nn.Conv2d(1, 1, k, s, p, dilation=d)(torch.zeros(1, 1, h, h)).shape[-1]
                    mismatches += real != conv_out(h, k, s, p, d)
print("mismatches:", mismatches)
print("7x7 through MaxPool2d(2):", nn.MaxPool2d(2)(torch.zeros(1, 1, 7, 7)).shape)

## 2. Three ways to see shapes

We use a slightly bigger CIFAR-style network as our running example (input $3\times32\times32$).

### 2.1 The quick-and-dirty way: print inside `forward`

In [ ]:
class DebugCNN(nn.Module):
    def __init__(self, verbose=False):
        super().__init__()
        self.verbose = verbose
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def log(self, name, x):
        if self.verbose:
            print(f"{name:<12} {tuple(x.shape)}")

    def forward(self, x):
        self.log("input", x)
        x = self.pool(F.relu(self.conv1(x))); self.log("block1", x)
        x = self.pool(F.relu(self.conv2(x))); self.log("block2", x)
        x = self.pool(F.relu(self.conv3(x))); self.log("block3", x)
        x = x.flatten(1);                     self.log("flatten", x)
        x = F.relu(self.fc1(x));              self.log("fc1", x)
        x = self.fc2(x);                      self.log("logits", x)
        return x

_ = DebugCNN(verbose=True)(torch.randn(2, 3, 32, 32))

**Tip:** always test with a batch size of at least 2. With batch size 1, some bugs (for example `x.view(-1)` instead of `x.flatten(1)`, or `squeeze()` removing the batch dimension) produce the right-looking shape by accident.

### 2.2 A reusable summary function with forward hooks

A **forward hook** is a function PyTorch calls right after a module's `forward` runs, receiving `(module, inputs, output)`. Hooks let you observe any model, including ones you did not write (like torchvision's ResNet), without editing its code. We will reuse this idea in Notebook 4 to visualise feature maps.

In [ ]:
def summary(model, input_size, batch=2):
    rows, hooks = [], []

    def hook(module, inp, out):
        n_params = sum(p.numel() for p in module.parameters(recurse=False))
        rows.append((module.__class__.__name__, tuple(out.shape[1:]), n_params))

    for m in model.modules():
        if len(list(m.children())) == 0:      # only leaf modules
            hooks.append(m.register_forward_hook(hook))
    with torch.no_grad():
        model(torch.zeros(batch, *input_size))
    for h in hooks:
        h.remove()                            # always remove hooks when done

    print(f"{'layer':<14}{'output shape':<22}{'params':>10}")
    print("-" * 46)
    for name, shape, n in rows:
        print(f"{name:<14}{str(shape):<22}{n:>10,}")
    print("-" * 46)
    print(f"{'total':<36}{sum(p.numel() for p in model.parameters()):>10,}")

seq_model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
    nn.Linear(256, 10),
)
summary(seq_model, (3, 32, 32))

### 2.3 Libraries

In practice, `pip install torchinfo` and `from torchinfo import summary; summary(model, (1, 3, 32, 32))` prints a polished version of the above, including MACs. It is worth knowing how it works (hooks), which is why we built our own first.

## 3. Where do the parameters live?

From the table above, notice something striking: the first fully connected layer, `Linear(2048, 256)`, holds about **524k** of the **~620k** parameters. All three conv layers together hold less than 100k.

This is typical of "classic" CNNs (VGG-16 has ~124M of its 138M parameters in its fully connected layers). Convolutions are parameter-cheap because of weight sharing; dense layers on top of a large flattened map are expensive. Modern architectures replace `flatten → big Linear` with **global average pooling** (Section 6), shrinking the head to $C \times \text{classes}$ parameters.

## 4. Compute cost: MACs

Parameters measure memory; **multiply-accumulate operations (MACs)** measure compute. For a conv layer, each output element needs $C_\text{in} k^2$ multiply-adds, and there are $C_\text{out} H_\text{out} W_\text{out}$ output elements:

$$
\text{MACs}_\text{conv} = C_\text{out}\, H_\text{out}\, W_\text{out}\, C_\text{in}\, k^2,
\qquad
\text{MACs}_\text{linear} = d_\text{in}\, d_\text{out}.
$$

So unlike parameters, conv compute **does** depend on resolution. Early, high-resolution layers are cheap in parameters but expensive in compute; it is the opposite for the dense head.

In [ ]:
def count_macs(model, input_size):
    macs = []
    def hook(m, inp, out):
        if isinstance(m, nn.Conv2d):
            c_out, h, w = out.shape[1:]
            kh, kw = m.kernel_size
            macs.append(("Conv2d", c_out * h * w * (m.in_channels // m.groups) * kh * kw))
        elif isinstance(m, nn.Linear):
            macs.append(("Linear", m.in_features * m.out_features))
    hooks = [m.register_forward_hook(hook) for m in model.modules()]
    with torch.no_grad():
        model(torch.zeros(1, *input_size))
    for h in hooks: h.remove()
    return macs

macs = count_macs(seq_model, (3, 32, 32))
params = [m for m in seq_model if isinstance(m, (nn.Conv2d, nn.Linear))]
names = [f"{n}{i}" for i, (n, _) in enumerate(macs)]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.bar(names, [sum(p.numel() for p in m.parameters()) for m in params]); a1.set_title("parameters per layer")
a2.bar(names, [v for _, v in macs], color="tab:orange"); a2.set_title("MACs per layer")
for a in (a1, a2): a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print(f"total MACs: {sum(v for _, v in macs) / 1e6:.1f} M")

## 5. Receptive fields

The **receptive field (RF)** of a unit is the region of the *input image* that can influence its value. In a single $3\times3$ conv the RF is $3\times3$. Stack two and each unit of the second layer sees $3\times3$ units of the first, each of which sees $3\times3$ pixels, overlapping: total $5\times5$.

The RF matters because a unit cannot detect a pattern larger than what it sees. To classify a digit, at least the final features must see most of the digit.

### Recursive formula

Process layers from input to output. Keep two numbers: the RF size $r$ and the **jump** $j$ (the distance in input pixels between two adjacent units of the current layer). Start with $r_0 = 1, j_0 = 1$. For a layer with kernel $k$, stride $s$, dilation $d$:

$$
r_\ell = r_{\ell-1} + d_\ell (k_\ell - 1)\, j_{\ell-1},
\qquad
j_\ell = j_{\ell-1}\, s_\ell.
$$

Intuition: the new kernel spans $k-1$ extra neighbours, each one $j_{\ell-1}$ input pixels apart.

Consequences:

* Stacking stride-1 $3\times3$ convs grows the RF **linearly** (by 2 per layer).
* Every downsampling (stride or pooling) **doubles the jump**, so later layers grow the RF much faster. This is the main reason CNNs downsample.
* Two $3\times3$ convs see a $5\times5$ region with $2\cdot 9 C^2 = 18C^2$ weights versus $25C^2$ for one $5\times5$, plus an extra non-linearity. This observation is the core design idea of VGG.

In [ ]:
def receptive_field(model, verbose=True):
    r, j = 1, 1
    if verbose:
        print(f"{'layer':<12}{'k':>3}{'s':>3}{'d':>3}{'RF':>6}{'jump':>6}")
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d, nn.AvgPool2d)):
            k = m.kernel_size if isinstance(m.kernel_size, int) else m.kernel_size[0]
            s = m.stride if isinstance(m.stride, int) else m.stride[0]
            d = getattr(m, "dilation", 1)
            d = d if isinstance(d, int) else d[0]
            r = r + d * (k - 1) * j
            j = j * s
            if verbose:
                print(f"{m.__class__.__name__:<12}{k:>3}{s:>3}{d:>3}{r:>6}{j:>6}")
    return r

rf = receptive_field(seq_model)
print(f"\nThe last conv features each see a {rf}x{rf} window of the 32x32 input.")

### Measuring the receptive field empirically

Theory is nice; let's check it. The gradient of one output unit with respect to the input is non-zero exactly on the pixels that influence it. So: pick the centre unit of a feature map, backpropagate from it alone, and plot $|\partial y / \partial x|$.

This also reveals the **effective receptive field**: the gradient is concentrated near the centre (roughly Gaussian), so in practice a unit "pays attention" to a region much smaller than its theoretical RF (Luo et al., 2016).

In [ ]:
import copy

def empirical_rf(feature_extractor, in_ch=3, size=64):
    fe = copy.deepcopy(feature_extractor)
    # 1) Constant positive weights so the map reflects architecture, not random init.
    # 2) Max pooling routes gradient to a single winner per window, which would hide
    #    part of the RF; average pooling has the same RF but spreads the gradient.
    for name, m in list(fe.named_children()):
        if isinstance(m, nn.MaxPool2d):
            setattr(fe, name, nn.AvgPool2d(m.kernel_size, m.stride, m.padding))
    for m in fe.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.constant_(m.weight, 1.0 / (m.in_channels * m.kernel_size[0] ** 2))
            nn.init.zeros_(m.bias)
    # Positive input keeps every ReLU active (ReLU has zero gradient at 0).
    x = torch.ones(1, in_ch, size, size, requires_grad=True)
    y = fe(x)
    c = y.shape[-1] // 2
    y[0, :, c, c].sum().backward()
    g = x.grad[0].abs().sum(0)
    nz = (g > 0).nonzero()
    extent = (nz.max(0).values - nz.min(0).values + 1).tolist()
    return g, extent

configs = {
    "3 convs, no pooling": nn.Sequential(*[l for _ in range(3) for l in (nn.Conv2d(3 if _ == 0 else 8, 8, 3, padding=1), nn.ReLU())]),
    "3 conv+pool blocks": nn.Sequential(*list(seq_model.children())[:8]),  # up to the 3rd conv + ReLU
    "dilated (d=1,2,4)": nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
                                       nn.Conv2d(8, 8, 3, padding=2, dilation=2), nn.ReLU(),
                                       nn.Conv2d(8, 8, 3, padding=4, dilation=4), nn.ReLU()),
}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, fe) in zip(axes, configs.items()):
    g, extent = empirical_rf(fe)
    theory = receptive_field(fe, verbose=False)
    ax.imshow(g, cmap="magma")
    ax.set_title(f"{name}\nmeasured {extent[0]}x{extent[1]}, theory {theory}x{theory}")
    ax.axis("off")
plt.tight_layout(); plt.show()

Note the three patterns:

* **No pooling:** RF grows by 2 per layer: $3, 5, 7$.
* **With pooling:** conv, pool, conv, pool, conv gives $3 \to 4 \to 8 \to 10 \to 18$. Each pool doubles the jump, so every later conv adds twice as much: large context, cheaply.
* **Dilation** $1,2,4$: RF $3 \to 7 \to 15$ without any downsampling, which is why dilated convs are popular in segmentation, where you must keep full resolution. The "gridded" pattern shows the risk: some pixels are skipped (gridding artefacts).

## 6. Removing hard-coded sizes

The number `128 * 4 * 4` in our model is fragile: change the input size to $64\times64$ and the model crashes. Three remedies:

### 6.1 Compute it with a dummy forward pass

In [ ]:
features = nn.Sequential(*list(seq_model.children())[:9])  # everything up to Flatten
with torch.no_grad():
    n_flat = features(torch.zeros(1, 3, 32, 32)).numel()
print("flattened size computed automatically:", n_flat)

### 6.2 `nn.LazyLinear`

`nn.LazyLinear(out_features)` infers `in_features` at the first forward call. Remember to run a dummy batch **before** creating the optimizer, otherwise the optimizer will not see the not-yet-created parameters.

In [ ]:
lazy = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                     nn.Flatten(), nn.LazyLinear(10))
print("before:", lazy[-1])
lazy(torch.zeros(2, 3, 32, 32))
print("after: ", lazy[-1])

### 6.3 Global average pooling (the modern way)

`nn.AdaptiveAvgPool2d(1)` averages each channel over all spatial positions, mapping $(N, C, H, W) \to (N, C, 1, 1)$ **for any** $H, W$. The head becomes `Linear(C, classes)`.

$$
g_c = \frac{1}{HW}\sum_{i,j} F[c, i, j].
$$

Benefits:

* Works for any input resolution.
* Far fewer parameters (for us: $128 \cdot 10$ instead of $2048 \cdot 256 + 256 \cdot 10$).
* Built-in translation invariance: the position of a feature no longer matters after averaging.
* Acts as a regulariser and makes class activation maps possible (Notebook 4).

ResNet, EfficientNet, MobileNet and most modern CNNs end this way.

In [ ]:
gap_model = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(128, 10),
)
summary(gap_model, (3, 32, 32))
for s in [32, 64, 97, 224]:
    print(f"input {s}x{s} -> output", tuple(gap_model(torch.zeros(2, 3, s, s)).shape))

## 7. A field guide to shape errors

Let's trigger the classics on purpose and read the messages.

In [ ]:
def show_error(title, fn):
    try:
        fn()
        print(f"[{title}] no error?!")
    except Exception as e:
        print(f"[{title}]\n  {type(e).__name__}: {str(e).splitlines()[0]}\n")

# 1. Wrong in_features in the Linear head
show_error("wrong flatten size",
           lambda: nn.Sequential(nn.Conv2d(3, 8, 3), nn.Flatten(), nn.Linear(8 * 32 * 32, 10))(torch.zeros(2, 3, 32, 32)))
# 2. Channel mismatch between consecutive convs
show_error("channel mismatch",
           lambda: nn.Sequential(nn.Conv2d(3, 16, 3), nn.Conv2d(32, 16, 3))(torch.zeros(2, 3, 32, 32)))
# 3. Wrong number of input channels
show_error("grayscale image fed to RGB model",
           lambda: nn.Conv2d(3, 8, 3)(torch.zeros(2, 1, 28, 28)))
# 4. Channels-last data (N,H,W,C), e.g. straight from NumPy/PIL
show_error("NHWC instead of NCHW",
           lambda: nn.Conv2d(3, 8, 3)(torch.zeros(2, 32, 32, 3)))
# 5. Image too small for the network
show_error("input too small",
           lambda: nn.Sequential(*[nn.MaxPool2d(2)] * 6, nn.Conv2d(3, 8, 3))(torch.zeros(2, 3, 32, 32)))

**How to read `mat1 and mat2 shapes cannot be multiplied (A x B and C x D)`**: `A` is the batch size, `B` is what actually arrived at the Linear layer, `C` is what the layer expected (`in_features`). Set `in_features = B`.

**Fixes for the others:** match `in_channels` to the previous layer's `out_channels`; convert grayscale to 3 channels with `x.repeat(1, 3, 1, 1)` or change the first conv; convert NHWC to NCHW with `x.permute(0, 3, 1, 2)`; use fewer downsampling stages or larger inputs.

## Summary

* $H_\text{out} = \lfloor (H + 2p - d(k-1) - 1)/s \rfloor + 1$. Same-padding for odd $k$: $p = (k-1)/2$.
* Forward hooks let you inspect any model without modifying it.
* Conv params are resolution-independent, conv compute is not. Dense heads on large flattened maps dominate parameter counts.
* RF recursion: $r \mathrel{+}= d(k-1)j$, $j \mathrel{*}= s$. Downsampling accelerates RF growth; the *effective* RF is smaller and Gaussian-like.
* Prefer `AdaptiveAvgPool2d(1)` over hard-coded flatten sizes.

## Exercises

1. By hand, compute the shapes, parameters and receptive field for: `Conv(3,64,7,s=2,p=3) → MaxPool(3,s=2,p=1) → Conv(64,64,3,p=1) → Conv(64,128,3,s=2,p=1)` on a $224\times224$ input. (This is the start of ResNet-18.) Then verify with `summary` and `receptive_field`.
2. Replace every `MaxPool2d(2)` in `seq_model` with a stride-2 conv. How do parameters, MACs and RF change?
3. What is the smallest input size that `seq_model` accepts? Explain using the formula, then test it.
4. Use `summary` on `torchvision.models.resnet18()` (no weights needed). Which layer has the most parameters? Which has the most MACs?
5. Using `empirical_rf`, plot how the measured RF grows as you stack 1 to 10 stride-1 $3\times3$ convs. Also plot the *standard deviation* of the gradient map: does it grow like $L$ or like $\sqrt{L}$? (The latter is the effective-RF result.)